# Phase 6 — Module 3 validation

Held-out datasets vs the two frozen rules. Predictions saved before training; scored after.


In [1]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
from virgo import frozen_rules as fr
from experiments import predict_module3 as pm, score_module3 as sm

# One table style for every table below: fixed layout + wrapped headers so a wide table never scrolls out of the output area.
STYLE = [
    {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed"), ("font-size", "11px")]},
    {"selector": "th", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
    {"selector": "td", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
]
fr.HELDOUT

/home/m-adam/miniconda/envs/i2v/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['citeseer_linqs',
 'proteins',
 'pubmed',
 'actor',
 'minesweeper',
 'amazon_photo',
 'lastfm_asia',
 'amazon_ratings',
 'squirrel_filtered']

## 1 · Predict — before training

Both frozen rules are **link-prediction** rules (Module 2 found no node-classification rule).


In [2]:
pred, added = pm.freeze_predictions(fr.HELDOUT)
print(f"newly frozen: {added or 'none (already saved before training)'}")

# Display only: a disagreement is decided by the LEAD rule (frozen_rules.LEAD = rule 1), so one verdict, one rule named.
NAMES = {"rule1": "R1 (adj_h)", "rule2": "R2 (largest_comp_frac)"}
split = pred["rule1_pred"] != pred["rule2_pred"]
lead = pred[f"{fr.LEAD.name}_pred"] + f"  ({NAMES[fr.LEAD.name]})"
shown = pred.drop(columns=["tasks"]).assign(predicted_verdict=pred["predicted_verdict"].where(~split, lead))

display(
    shown.round(4)
    .rename(columns={
        "homophily_adjusted": "adj_h",
        "rule1_interval": "R1 (adj_h) interval",
        "rule1_pred": "R1 (adj_h) prediction",
        "largest_component_frac": "largest_comp_frac",
        "rule2_interval": "R2 (largest_comp_frac) interval",
        "rule2_pred": "R2 (largest_comp_frac) prediction",
        "predicted_verdict": "predicted LP verdict",
    })
    .style
    .format(na_rep="—")
    .hide(axis="index")
    .set_table_styles(STYLE)
)

newly frozen: none (already saved before training)


dataset,domain,adj_h,R1 (adj_h) interval,R1 (adj_h) prediction,largest_comp_frac,R2 (largest_comp_frac) interval,R2 (largest_comp_frac) prediction,predicted LP verdict
citeseer_linqs,citation,0.673100,"(0.0926, 0.3613)",keep original,0.646400,"(0.9177, 1.0)",keep original,keep original
proteins,biological,0.355200,"(0.0926, 0.3613)",keep original,0.014300,"(0.9177, 1.0)",keep original,keep original
pubmed,citation,0.686000,"(0.0926, 0.3613)",keep original,1.000000,"(0.9177, 1.0)",augment,keep original (R1 (adj_h))
actor,film,0.002800,"(0.0926, 0.3613)",augment,1.000000,"(0.9177, 1.0)",augment,augment
minesweeper,grid,0.009400,"(0.0926, 0.3613)",augment,1.000000,"(0.9177, 1.0)",augment,augment
amazon_photo,co-purchase,0.785000,"(0.0926, 0.3613)",keep original,0.978700,"(0.9177, 1.0)",augment,keep original (R1 (adj_h))
lastfm_asia,music social,0.856200,"(0.0926, 0.3613)",keep original,1.000000,"(0.9177, 1.0)",augment,keep original (R1 (adj_h))
amazon_ratings,co-purchase,0.140200,"(0.0926, 0.3613)",augment,1.000000,"(0.9177, 1.0)",augment,augment
squirrel_filtered,wikipedia,0.008600,"(0.0926, 0.3613)",augment,1.000000,"(0.9177, 1.0)",augment,augment


## 2 · Verdict vs actual — after training


In [3]:
scored = sm.score(fr.HELDOUT)
scored.to_csv(sm.SCORED_CSV, index=False)
for r in fr.FROZEN_RULES:
    col = list(scored[f"{r.name}_correct"])
    c = [v for v in col if isinstance(v, bool)]
    skipped = sorted({str(v) for v in col if not isinstance(v, bool)})
    print(f"{r.name} ({r.predictor} {r.op} {r.point}): "
          + (f"{sum(c)}/{len(c)} correct" if c else "nothing scored yet")
          + (f"  [not scored: {', '.join(skipped)}]" if skipped else ""))

# One-line scoreboard: how many held-out datasets, and how many each rule called right.
hits = {r.name: [v for v in scored[f"{r.name}_correct"] if isinstance(v, bool)] for r in fr.FROZEN_RULES}
print(f"STATS  datasets {len(scored)}  |  R1 (adj_h) {sum(hits['rule1'])}/{len(hits['rule1'])} correct  |  "
      f"R2 (largest_comp_frac) {sum(hits['rule2'])}/{len(hits['rule2'])} correct  |  "
      f"{len(scored) - len(hits['rule1'])} not scored (tie / pending)")

# Display only: same lead-rule tie-break as cell 1 - one verdict, one rule named.
NAMES = {"rule1": "R1 (adj_h)", "rule2": "R2 (largest_comp_frac)"}
split = scored["rule1_pred"] != scored["rule2_pred"]
lead = scored[f"{fr.LEAD.name}_pred"] + f"  ({NAMES[fr.LEAD.name]})"
shown = (scored.drop(columns=["rule1_correct", "rule2_correct"])   # per-rule accuracy printed above; the CSV keeps both
         .assign(predicted_verdict=scored["predicted_verdict"].where(~split, lead)))

display(
    shown.round(4)
    .rename(columns={
        "homophily_adjusted": "adj_h",
        "largest_component_frac": "largest_comp_frac",
        "rule1_pred": "R1 (adj_h) prediction",
        "rule2_pred": "R2 (largest_comp_frac) prediction",
        "predicted_verdict": "predicted LP verdict",
        "actual_verdict": "actual LP verdict",
        "best_augmented": "best aug",
        "best_variant": "best variant",
    })
    .style
    .format(na_rep="—")
    .hide(axis="index")
    .set_table_styles(STYLE)
)

rule1 (homophily_adjusted < 0.227): 5/7 correct  [not scored: no decision]
rule2 (largest_component_frac > 0.9588): 4/7 correct  [not scored: no decision]
STATS  datasets 9  |  R1 (adj_h) 5/7 correct  |  R2 (largest_comp_frac) 4/7 correct  |  2 not scored (tie / pending)


dataset,adj_h,largest_comp_frac,R1 (adj_h) prediction,R2 (largest_comp_frac) prediction,predicted LP verdict,actual LP verdict,original,best aug,best variant
citeseer_linqs,0.673100,0.646400,keep original,keep original,keep original,keep original,0.621800,0.543700,centrality
proteins,0.355200,0.014300,keep original,keep original,keep original,keep original,0.672000,0.583400,centrality
pubmed,0.686000,1.000000,keep original,augment,keep original (R1 (adj_h)),tie,0.639200,0.623800,hybrid
actor,0.002800,1.000000,augment,augment,augment,augment,0.595300,0.661700,degree
minesweeper,0.009400,1.000000,augment,augment,augment,keep original,0.709300,0.667100,hybrid
amazon_photo,0.785000,0.978700,keep original,augment,keep original (R1 (adj_h)),tie,0.785700,0.780300,hybrid
lastfm_asia,0.856200,1.000000,keep original,augment,keep original (R1 (adj_h)),keep original,0.721800,0.700200,hybrid
amazon_ratings,0.140200,1.000000,augment,augment,augment,keep original,0.753600,0.655000,hybrid
squirrel_filtered,0.008600,1.000000,augment,augment,augment,augment,0.739900,0.773600,degree


## 3 · Held-out evidence

Every Module-3 number in one table: frozen property and prediction beside what training measured.


In [4]:
ev = sm.evidence(fr.HELDOUT)
ev.to_csv(sm.EVIDENCE_CSV, index=False)

wide = pd.DataFrame({
    "dataset": ev["dataset"],
    "adj_h": ev["homophily_adjusted"].round(4),
    "largest_comp_frac": ev["largest_component_frac"].round(4),
    "R1 (adj_h) prediction": ev["rule1_pred"],
    "R2 (largest_comp_frac) prediction": ev["rule2_pred"],
    "original mean ± std": [f"{m:.4f} ± {s:.4f}" if m == m else "—" for m, s in zip(ev["original"], ev["original_std"])],
    "best aug mean ± std": [f"{m:.4f} ± {s:.4f}" if m == m else "—" for m, s in zip(ev["best_augmented"], ev["best_augmented_std"])],
    "best variant": ev["best_variant"],
    "abs gap": ev["gap_abs"],
    "seed noise": ev["noise"],
    "gap / noise": ev["gap_sigma"],
    "actual LP verdict": ev["actual_verdict"],
    "R1 correct": ev["rule1_correct"].astype(str),
    "R2 correct": ev["rule2_correct"].astype(str),
})
display(wide.style.format(na_rep="—").hide(axis="index").set_table_styles(STYLE))

dataset,adj_h,largest_comp_frac,R1 (adj_h) prediction,R2 (largest_comp_frac) prediction,original mean ± std,best aug mean ± std,best variant,abs gap,seed noise,gap / noise,actual LP verdict,R1 correct,R2 correct
citeseer_linqs,0.673100,0.646400,keep original,keep original,0.6218 ± 0.0255,0.5437 ± 0.0165,centrality,-0.078100,0.021500,-3.640000,keep original,True,True
proteins,0.355200,0.014300,keep original,keep original,0.6720 ± 0.0040,0.5834 ± 0.0036,centrality,-0.088600,0.003800,-23.280000,keep original,True,True
pubmed,0.686000,1.000000,keep original,augment,0.6392 ± 0.0337,0.6238 ± 0.0063,hybrid,-0.015400,0.024200,-0.640000,tie,no decision,no decision
actor,0.002800,1.000000,augment,augment,0.5953 ± 0.0307,0.6617 ± 0.0068,degree,0.066400,0.022200,2.990000,augment,True,True
minesweeper,0.009400,1.000000,augment,augment,0.7093 ± 0.0089,0.6671 ± 0.0046,hybrid,-0.042200,0.007100,-5.960000,keep original,False,False
amazon_photo,0.785000,0.978700,keep original,augment,0.7857 ± 0.0249,0.7803 ± 0.0270,hybrid,-0.005400,0.026000,-0.210000,tie,no decision,no decision
lastfm_asia,0.856200,1.000000,keep original,augment,0.7218 ± 0.0251,0.7002 ± 0.0086,hybrid,-0.021600,0.018800,-1.150000,keep original,True,False
amazon_ratings,0.140200,1.000000,augment,augment,0.7536 ± 0.0204,0.6550 ± 0.0116,hybrid,-0.098600,0.016600,-5.940000,keep original,False,False
squirrel_filtered,0.008600,1.000000,augment,augment,0.7399 ± 0.0114,0.7736 ± 0.0033,degree,0.033700,0.008400,4.020000,augment,True,True


## 4 · Frozen discovery evidence

Read-only from `results/candidate_rules.csv` (Module 2 / notebook 5). No threshold is computed here.


In [5]:
disc = sm.discovery_evidence()          # asserts the file still carries the frozen thresholds; never re-fits

display(
    disc.rename(columns={
        "predictor": "predictor (graph property)",
        "threshold": "fixed threshold",
        "interval": "threshold interval",
        "spearman_rho": "Spearman rho (best variant)",
        "fixed_variant_rho": "Spearman rho (fixed variant)",
        "lodo_min_abs_rho": "LODO min |rho|",
        "n_datasets": "datasets",
        "n_exceptions": "exceptions",
    })
    .style.format(na_rep="—").hide(axis="index").set_table_styles(STYLE)
)

rule,predictor (graph property),fixed threshold,threshold interval,Spearman rho (best variant),Spearman rho (fixed variant),LODO min |rho|,datasets,exceptions
rule1,homophily_adjusted,0.227000,"(0.0926, 0.3613)",-0.900000,-1.000000,0.800000,5,0
rule2,largest_component_frac,0.958800,"(0.9177, 1.0)",0.777500,0.777500,0.707100,6,0


## 5 · Findings


In [6]:
hits = {r.name: [v for v in ev[f"{r.name}_correct"] if isinstance(v, bool)] for r in fr.FROZEN_RULES}
nc = sm.nc_verdicts(fr.HELDOUT)                     # no rule covers node classification - reported as the boundary check
nc_ok = nc[nc["usable"]]
d = disc.set_index("rule")
nc_disc = pd.read_csv(sm.DISCOVERY_CSV).query("task_family == 'node classification' and target == 'gap_rel'").iloc[0]
split_rules, tied = ev["rule1_pred"] != ev["rule2_pred"], ev["actual_verdict"] == "tie"

findings = pd.DataFrame([
    {"finding": "Adjusted homophily predicts LP augmentation (R1)",
     "evidence": f"discovery rho {d.loc['rule1', 'spearman_rho']} (fixed variant {d.loc['rule1', 'fixed_variant_rho']}), "
                 f"LODO min |rho| {d.loc['rule1', 'lodo_min_abs_rho']}, {d.loc['rule1', 'n_datasets']} datasets; "
                 f"held-out {sum(hits['rule1'])}/{len(hits['rule1'])} correct",
     "limitation": "minesweeper is the counterexample: heterophilous AND augmented by the rule, but structurally uniform "
                   "(3 distinct degrees), so role edges replace real adjacency and it loses"},
    {"finding": "Fragmentation predicts LP augmentation (R2)",
     "evidence": f"discovery rho {d.loc['rule2', 'spearman_rho']} (fixed variant {d.loc['rule2', 'fixed_variant_rho']}), "
                 f"LODO min |rho| {d.loc['rule2', 'lodo_min_abs_rho']}, {d.loc['rule2', 'n_datasets']} datasets; "
                 f"held-out {sum(hits['rule2'])}/{len(hits['rule2'])} correct",
     "limitation": "two-group split - every discovery augment cell sits at largest_comp_frac 1.0, so the cut is arbitrary "
                   "inside its interval and still coincides with dataset provenance; misses minesweeper too"},
    {"finding": "Node classification has no augmentation rule",
     "evidence": f"discovery {int(nc_disc['n_augment'])}/{int(nc_disc['n_decided'])} decided NC cells augment; "
                 f"held-out {int((nc_ok['verdict'] == 'augment').sum())}/{len(nc_ok)}",
     "limitation": "a boundary, not a rule: with no augment case there is nothing to separate, so more NC datasets are needed"},
    {"finding": f"{int((split_rules & tied).sum())} of {int(split_rules.sum())} rule disagreements came back a tie",
     "evidence": f"{int(tied.sum())} of {len(ev)} held-out datasets tie, all in the homophilous + connected quadrant "
                 "where the best variant is hybrid (additive, so it keeps every original edge)",
     "limitation": "a tie scores neither rule, so the disagreement quadrant is still untested"},
])

display(findings.rename(columns={"finding": "Finding", "evidence": "Evidence", "limitation": "Limitation"})
        .style.hide(axis="index").set_table_styles(STYLE)
        .set_properties(**{"text-align": "left"}))

Finding,Evidence,Limitation
Adjusted homophily predicts LP augmentation (R1),"discovery rho -0.9 (fixed variant -1.0), LODO min |rho| 0.8, 5 datasets; held-out 5/7 correct","minesweeper is the counterexample: heterophilous AND augmented by the rule, but structurally uniform (3 distinct degrees), so role edges replace real adjacency and it loses"
Fragmentation predicts LP augmentation (R2),"discovery rho 0.7775 (fixed variant 0.7775), LODO min |rho| 0.7071, 6 datasets; held-out 4/7 correct","two-group split - every discovery augment cell sits at largest_comp_frac 1.0, so the cut is arbitrary inside its interval and still coincides with dataset provenance; misses minesweeper too"
Node classification has no augmentation rule,discovery 0/4 decided NC cells augment; held-out 0/8,"a boundary, not a rule: with no augment case there is nothing to separate, so more NC datasets are needed"
2 of 3 rule disagreements came back a tie,"2 of 9 held-out datasets tie, all in the homophilous + connected quadrant where the best variant is hybrid (additive, so it keeps every original edge)","a tie scores neither rule, so the disagreement quadrant is still untested"


## 6 · Rule redundancy diagnostics

- Review point 1: property–property Spearman between the two rule inputs across **all datasets** (7 discovery + 9 held-out) — is R2 redundant with R1 by construction?
- Review point 2: the 2×2 agreement table of R1 × R2 decisions over **every dataset**, and each case where they disagree.
- Descriptive only — nothing is refitted; reads the frozen property tables / predictions / scoreboard. Discovery rows are in-sample (the rules were fitted there), so their correctness describes fit, not validation.


In [7]:
# Review point 1: property-property Spearman across ALL datasets - 7 discovery + 9 held-out (review request 2026-08-08:
# all datasets, not held-out only). ogbl_ddi is unlabelled -> no homophily, so the pair rho uses the 15 complete pairs.
# Off-diagonal = the R1-vs-R2 correlation; n_at_1.0 shows R2's ceiling degeneracy.
props = sm.rule_properties()
display(props.sort_values("homophily_adjusted").style.format(precision=4, na_rep="—").hide(axis="index").set_table_styles(STYLE))
corr = sm.property_correlation()
corr.to_csv(sm.CORR_CSV)
display(corr.reset_index().style.format(precision=4).hide(axis="index").set_table_styles(STYLE))

# Review point 2 - THE 2x2 AGREEMENT TABLE over EVERY dataset (7 discovery + 9 held-out). Off-diagonal = disagreements.
# ogbl_ddi drops out of the 2x2: unlabelled -> R1 cannot fire, a coverage hole, not a disagreement.
agree = sm.decision_agreement()
agree.to_csv(sm.AGREE_CSV, index=False)
order = ["augment", "keep original"]
xtab = (pd.crosstab(agree["rule1_pred"], agree["rule2_pred"]).reindex(index=order, columns=order, fill_value=0)
        .rename(index=lambda v: f"R1: {v}", columns=lambda v: f"R2: {v}").rename_axis(index="", columns=""))
display(xtab.reset_index().rename(columns={"": " "}).style.hide(axis="index").set_table_styles(STYLE))

# Per-dataset detail behind the 2x2: panel | R1 prediction | R2 prediction | actual result | which rule was correct.
# Discovery rows are IN-SAMPLE (rules fitted there) - their "correct" describes fit, not validation. A tie scores
# neither rule, but the gap still has a sign - credit the rule on the marginally-better side of it.
sig = sm.actual_lp_verdicts(agree["dataset"].tolist()).set_index("dataset")["gap_sigma"]
def which(ds, p1, p2, r1, r2):
    if "n/a" in (p1, p2): return "— (R1 n/a: no labels)"
    if (r1, r2) == (True, True): return "both correct"
    if (r1, r2) == (False, False): return "both wrong"
    if r1 is True: return "R1"
    if r2 is True: return "R2"
    s = sig.get(ds, float("nan"))
    if s != s: return "— (no LP cell)"
    lean = "keep original" if s < 0 else "augment"             # tie: inside seed noise, only the sign leans
    return "R1 (marginally)" if p1 == lean != p2 else "R2 (marginally)" if p2 == lean != p1 else "neither (tie)"
shown = pd.DataFrame({
    "dataset": agree["dataset"],
    "panel": agree["panel"],
    "R1 prediction": agree["rule1_pred"],
    "R2 prediction": agree["rule2_pred"],
    "actual result": agree["actual_verdict"],
    "rules agree?": agree["rules_agree"].map({True: "agree", False: "DISAGREE"}).fillna("R1 n/a"),
    "which rule was correct": [which(d, p1, p2, r1, r2) for d, p1, p2, r1, r2 in
                               zip(agree["dataset"], agree["rule1_pred"], agree["rule2_pred"],
                                   agree["rule1_correct"], agree["rule2_correct"])],
})
display(shown.style.hide(axis="index").set_table_styles(STYLE))

# The counts: how often the rules agree, how often they disagree, and who wins the decided disagreements.
dis = agree[agree["rules_agree"] == False]                     # == because the column holds True/False/None (R1 n/a)
r1w = int((dis["disagreement_result"] == "R1 correct, R2 wrong").sum())
r2w = int((dis["disagreement_result"] == "R2 correct, R1 wrong").sum())
print(f"COUNTS  {len(agree)} datasets ({int(agree['rules_agree'].isna().sum())} R1-n/a)  |  "
      f"agree {int((agree['rules_agree'] == True).sum())}  |  disagree {len(dis)}  |  "
      f"in disagreements: R1 correct {r1w}, R2 correct {r2w}, undecided {len(dis) - r1w - r2w}")

dataset,panel,homophily_adjusted,largest_component_frac
roman_empire,discovery,-0.0468,1.0000
actor,held-out,0.0028,1.0000
squirrel_filtered,held-out,0.0086,1.0000
minesweeper,held-out,0.0094,1.0000
questions,discovery,0.0207,1.0000
tolokers,discovery,0.0926,1.0000
amazon_ratings,held-out,0.1402,1.0000
proteins,held-out,0.3552,0.0143
enzymes,discovery,0.3613,0.0064
ogbn_arxiv,discovery,0.5877,1.0000


property,homophily_adjusted,largest_component_frac,n_values,distinct_values,n_at_1.0,n_datasets,spearman_p
homophily_adjusted,1.0000,-0.4146,15,15,0,15,0.1244
largest_component_frac,-0.4146,1.0000,16,6,11,15,0.1244


,R2: augment,R2: keep original
R1: augment,7,0
R1: keep original,4,4


dataset,panel,R1 prediction,R2 prediction,actual result,rules agree?,which rule was correct
cora,discovery,keep original,keep original,keep original,agree,both correct
enzymes,discovery,keep original,keep original,keep original,agree,both correct
ogbn_arxiv,discovery,keep original,augment,no LP cell,DISAGREE,— (no LP cell)
ogbl_ddi,discovery,n/a,augment,augment,R1 n/a,— (R1 n/a: no labels)
roman_empire,discovery,augment,augment,augment,agree,both correct
tolokers,discovery,augment,augment,augment,agree,both correct
questions,discovery,augment,augment,augment,agree,both correct
citeseer_linqs,held-out,keep original,keep original,keep original,agree,both correct
proteins,held-out,keep original,keep original,keep original,agree,both correct
pubmed,held-out,keep original,augment,tie,DISAGREE,R1 (marginally)


COUNTS  16 datasets (1 R1-n/a)  |  agree 11  |  disagree 4  |  in disagreements: R1 correct 1, R2 correct 0, undecided 3
